# json

对应 `stdlib.md`：读写 JSON。

笔记本在 `python_base/json/qa.ipynb`。需要读写文件时，工作空间就是这个目录，题目文件落在旁边的子目录里。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录，即 python_base/json。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "json" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "python_base" / "json"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/python_base/json')

## 1. 字典和文本互转

把 `data` 变成 JSON 文本，再从这段文本读回字典。打印文本，以及读回来的 `name`。文本里的中文要原样可见。


In [5]:
data = {"name": "小明", "score": 3}

# 作答

import json

text = json.dumps(data, ensure_ascii=False, indent=4)

print(f"JSON 文本: {text}")

data_loaded = json.loads(text)

print(f"读回的字典 .name: {data_loaded['name']}")
#评阅
# 对。ensure_ascii=False 让「小明」原样出现，loads 之后 name 也是小明。

#参考答案
# import json
# text = json.dumps(data, ensure_ascii=False)
# print(text)
# print(json.loads(text)["name"])


JSON 文本: {
    "name": "小明",
    "score": 3
}
读回的字典 .name: 小明


## 2. 写进文件再读出来

把 `data` 写进 `box/user.json`，再从这个文件读回来。打印读到的 `age`。文件里的内容要带缩进，打开就能看。


In [6]:
box = ROOT / "q2"
box.mkdir(parents=True, exist_ok=True)
path = box / "user.json"
data = {"name": "小明", "age": 20}

# 作答

import json

path.write_text(
    json.dumps(data, ensure_ascii=False, indent=4),
    encoding="utf-8"
)

text = path.read_text(encoding="utf-8")

print(f"读回的文本: {text}")

data_loaded = json.loads(text)

print(f"读回的字典 .age: {data_loaded['age']}")
#评阅
# 对。文件里有缩进，读回的 age 是 20，中文也在。
# 写文件、读文件可以用 dump 和 load，直接对文件对象读写，不必先变成整段字符串。

#参考答案
# import json
# with path.open("w", encoding="utf-8") as handle:
#     json.dump(data, handle, ensure_ascii=False, indent=4)
# with path.open(encoding="utf-8") as handle:
#     print(json.load(handle)["age"])


读回的文本: {
    "name": "小明",
    "age": 20
}
读回的字典 .age: 20


## 3. 文本不合法

`raw` 不是合法的 JSON。解析它，不要让这一格中断。打印接到的异常类型名。


In [9]:
raw = '{"a": 1,}'

# 作答

import json

try:
    data_loaded = json.loads(raw)
except json.JSONDecodeError as e:
    print(f"异常类型名: {type(e).__name__},  {e.msg}")
#评阅
# 对。接住了 JSONDecodeError，这一格没有中断。类型名就是要打印的结果。

#参考答案
# import json
# try:
#     json.loads(raw)
# except json.JSONDecodeError as exc:
#     print(type(exc).__name__)


异常类型名: JSONDecodeError,  Illegal trailing comma before end of object


## 4. 一组对象

`data` 是一个列表。把它变成 JSON 文本，再读回来。打印第二项的 `name`。


In [10]:
data = [{"name": "小明"}, {"name": "小红"}]

# 作答

import json

text = json.dumps(data, ensure_ascii=False, indent=4)

print(f"JSON 文本: {text}")

data_loaded = json.loads(text)

print(f"读回的字典 .name: {data_loaded[1]['name']}")
#评阅
# 对。列表照样 dumps / loads，第二项的 name 是小红。

#参考答案
# import json
# loaded = json.loads(json.dumps(data, ensure_ascii=False))
# print(loaded[1]["name"])


JSON 文本: [
    {
        "name": "小明"
    },
    {
        "name": "小红"
    }
]
读回的字典 .name: 小红


## 5. 键按字母序

把 `data` 变成 JSON 文本，键按字母序排列。打印这段文本。


In [11]:
data = {"b": 1, "a": 2}

# 作答

import json

text = json.dumps(data, ensure_ascii=False, indent=4, sort_keys=True)

print(f"JSON 文本: {text}")
#评阅
# 对。sort_keys=True 之后 a 在 b 前面。

#参考答案
# import json
# print(json.dumps(data, sort_keys=True))


JSON 文本: {
    "a": 2,
    "b": 1
}


## 6. 日期不能直接写进去

`at` 是一个 `datetime`，不能直接变成 JSON。写成 ISO 格式的字符串后再输出文本。打印文本，里面要能看到 `2026-09-23`。


In [13]:
from datetime import datetime

data = {"at": datetime(2026, 9, 23, 9, 0)}

# 作答

import json

text = json.dumps(data, ensure_ascii=False, indent=4, default=str)

print(f"JSON 文本: {text}")
#评阅
# 文本里有 2026-09-23，但 default=str 得到的是「2026-09-23 09:00:00」，中间是空格。
# ISO 格式用 isoformat，中间是 T：2026-09-23T09:00:00。

#参考答案
# import json
# text = json.dumps(data, default=lambda value: value.isoformat())
# print(text)


JSON 文本: {
    "at": "2026-09-23 09:00:00"
}


## 7. 一行一个 JSON

`events.jsonl` 里一行一个对象。按文件里的顺序读回来，依次打印 `name`。


In [15]:
box = ROOT / "q7"
box.mkdir(parents=True, exist_ok=True)
path = box / "events.jsonl"
path.write_text('{"name": "小明"}\n{"name": "小红"}\n')

# 作答

import json

with path.open("r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        print(f"name: {data['name']}")
#评阅
# 对。一行 loads 一次，顺序是小明、小红。

#参考答案
# import json
# for line in path.read_text(encoding="utf-8").splitlines():
#     print(json.loads(line)["name"])


name: 小明
name: 小红


## 8. 读进来时收成坐标

`raw` 是一个对象。解析时把它收成 `(x, y)` 这样的二元组，打印这个二元组。


In [ ]:
raw = '{"x": 3, "y": 4}'

# 作答

import json

data = json.loads(raw)

print(f"二元组: {tuple(data.values())}")
#评阅
# 打印出来是 (3, 4)。这是先 loads 成 dict，再按 values() 的顺序取。
# 键如果换成先 y 后 x，结果就反了。解析时转换用 object_hook，并按键名取 x 和 y。

#参考答案
# import json
# def as_point(obj):
#     return (obj["x"], obj["y"])
# print(json.loads(raw, object_hook=as_point))
